# House Prices Baseline

This notebook builds a small regression baseline for the House Prices dataset. The goal is to practice a clear ML workflow before trying bigger feature engineering.

## 1. Import Libraries

We use pandas for loading data, scikit-learn for preprocessing and modeling, and matplotlib for a simple prediction plot.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## 2. Load the Data

Place the House Prices training file at `02_house_prices/data/train.csv`. The path cell below works whether the notebook runs from this folder or from the repository root.

In [ ]:
DATA_PATH = Path("data/train.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("02_house_prices/data/train.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError("Place train.csv in 02_house_prices/data/ before running this notebook.")

houses = pd.read_csv(DATA_PATH)
houses.head()

## 3. Choose a Small Feature Set

Start with features that are easy to understand. After the baseline runs, add more columns and compare metrics.

In [ ]:
target_column = "SalePrice"
numeric_features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "GarageArea",
    "TotalBsmtSF",
    "FullBath",
    "YearBuilt",
]
categorical_features = [
    "Neighborhood",
    "HouseStyle",
    "ExterQual",
    "KitchenQual",
]
feature_columns = numeric_features + categorical_features

X = houses[feature_columns]
y = houses[target_column]

X.head()

## 4. Split into Train and Test Sets

The test set gives a first estimate of how well the model handles rows it did not train on.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

len(X_train), len(X_test)

## 5. Build the Preprocessing and Model Pipeline

A pipeline keeps preprocessing and modeling together, which helps prevent mistakes when training and evaluating.

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(n_estimators=200, random_state=42)),
    ]
)

## 6. Train and Evaluate

MAE is easy to read because it is in dollars. RMSE punishes large errors more strongly. R2 gives a rough sense of explained variation.

In [ ]:
model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions) ** 0.5
r2 = r2_score(y_test, predictions)

print(f"MAE:  {mae:,.0f}")
print(f"RMSE: {rmse:,.0f}")
print(f"R2:   {r2:.3f}")

In [ ]:
plt.scatter(y_test, predictions, alpha=0.6)
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.title("House Prices Baseline: Actual vs Predicted")
plt.tight_layout()
plt.show()

## Next Experiments

- Try `LinearRegression` or `Ridge` and compare results.
- Add more numeric features one group at a time.
- Look at the rows with the largest prediction errors.